# IDUN raw sample reconstruction

This device-processing step matches each 20-sample XDF chunk to the native IDUN CSV and restores native acquisition order. It does **not** estimate a global IDUN → LSL clock mapping. Reconstruction QC is step 04 and clock alignment is step 05 under `../SyncQuality/`.

In [1]:
import importlib
import sys
from pathlib import Path

from IPython.display import display

for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (PROJECT_ROOT / 'helpers').is_dir():
        sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise FileNotFoundError('Could not locate the BIGPROJECT/helpers folder')

from helpers.idun import raw_lsl_repair as repair
from helpers.project_config import load_project_config
importlib.reload(repair)

CONFIG = load_project_config(PROJECT_ROOT / 'ImBrAid' / 'config.json')
PROJECT_NAME = CONFIG.project
DATA_ROOT = CONFIG.path('paths', 'processing_data_root')
OUTPUT_ROOT = CONFIG.path('paths', 'idun_repair_root')
PARTICIPANTS = None
VISITS = None
CHUNK_SAMPLES = 20
SEARCH_RADIUS_S = 30.0

In [2]:
jobs = repair.discover_idun_jobs(
    DATA_ROOT, project=PROJECT_NAME, participants=PARTICIPANTS, visits=VISITS,
)
display(jobs)

,project,pid,visit,discovery_status,raw_candidate_count,xdf_candidate_count,raw_csv_path,xdf_path
0,ImBrAid,A7G5SM,VX,missing_raw,0,1,None,E:\test\time_shifted\A7G5SM\LSL\A7G5SM_VX_lsl.xdf
1,ImBrAid,ABVLGQ,VX,ready,1,1,E:\test\time_shifted\ABVLGQ\IDUN\ABVLGQ_VX_eeg...,E:\test\time_shifted\ABVLGQ\LSL\ABVLGQ_VX_lsl.xdf
2,ImBrAid,B9NYQ8,VX,missing_raw,0,1,None,E:\test\time_shifted\B9NYQ8\LSL\B9NYQ8_VX_lsl.xdf
3,ImBrAid,CRVWF9,VX,ready,1,1,E:\test\time_shifted\CRVWF9\IDUN\CRVWF9_VX_eeg...,E:\test\time_shifted\CRVWF9\LSL\CRVWF9_VX_lsl.xdf
4,ImBrAid,CRYSEP,VX,ready,1,1,E:\test\time_shifted\CRYSEP\IDUN\CRYSEP_VX_eeg...,E:\test\time_shifted\CRYSEP\LSL\CRYSEP_VX_lsl.xdf
5,ImBrAid,CU6DM4,VX,ready,1,1,E:\test\time_shifted\CU6DM4\IDUN\CU6DM4_VX_eeg...,E:\test\time_shifted\CU6DM4\LSL\CU6DM4_VX_lsl.xdf
6,ImBrAid,DUJL2L,VX,ready,1,1,E:\test\time_shifted\DUJL2L\IDUN\DUJL2L_VX_eeg...,E:\test\time_shifted\DUJL2L\LSL\DUJL2L_VX_lsl.xdf
7,ImBrAid,E4MKXT,VX,ready,1,1,E:\test\time_shifted\E4MKXT\IDUN\E4MKXT_VX_eeg...,E:\test\time_shifted\E4MKXT\LSL\E4MKXT_VX_lsl.xdf
8,ImBrAid,EASFXM,VX,ready,1,1,E:\test\time_shifted\EASFXM\IDUN\EASFXM_VX_eeg...,E:\test\time_shifted\EASFXM\LSL\EASFXM_VX_lsl.xdf
9,ImBrAid,FEDZAG,VX,ready,1,1,E:\test\time_shifted\FEDZAG\IDUN\FEDZAG_VX_eeg...,E:\test\time_shifted\FEDZAG\LSL\FEDZAG_VX_lsl.xdf


In [3]:
summary, issues = repair.run_idun_repairs(
    jobs, OUTPUT_ROOT, chunk_samples=CHUNK_SAMPLES,
    search_radius_s=SEARCH_RADIUS_S,
)
display(summary[[
    'pid', 'visit', 'repair_status', 'usable_relative_time',
    'quality_grade', 'review_required', 'matched_chunk_ratio', 'warnings',
]])
display(issues)

,pid,visit,repair_status,usable_relative_time,quality_grade,review_required,matched_chunk_ratio,warnings
0,A7G5SM,VX,unavailable,False,D,True,NaN,Discovery status: missing_raw
1,ABVLGQ,VX,repaired,True,A,False,1.0,18 chunk transitions were not in raw acquisiti...
2,B9NYQ8,VX,unavailable,False,D,True,NaN,Discovery status: missing_raw
3,CRVWF9,VX,repaired,True,A,False,1.0,272 chunk transitions were not in raw acquisit...
4,CRYSEP,VX,repaired,True,A,False,1.0,120 chunk transitions were not in raw acquisit...
5,CU6DM4,VX,repaired,True,A,False,1.0,364 chunk transitions were not in raw acquisit...
6,DUJL2L,VX,repaired,True,A,False,1.0,193 chunk transitions were not in raw acquisit...
7,E4MKXT,VX,repaired,True,A,False,1.0,424 chunk transitions were not in raw acquisit...
8,EASFXM,VX,repaired,True,A,False,1.0,295 chunk transitions were not in raw acquisit...
9,FEDZAG,VX,repaired,True,A,False,1.0,295 chunk transitions were not in raw acquisit...


,project,pid,visit,severity,issue,detail
0,ImBrAid,A7G5SM,VX,error,source_discovery_failed,Discovery status: missing_raw
1,ImBrAid,ABVLGQ,VX,warning,xdf_chunk_reordering,18 adjacent XDF chunk transitions were not chr...
2,ImBrAid,ABVLGQ,VX,warning,nonmonotonic_xdf_timestamps,2342 uncorrected XDF sample intervals were non...
3,ImBrAid,ABVLGQ,VX,info,global_lsl_offset_unresolved,The repaired file uses IDUN raw time and conta...
4,ImBrAid,B9NYQ8,VX,error,source_discovery_failed,Discovery status: missing_raw
...,...,...,...,...,...,...
91,ImBrAid,YHYKDQ,VX,info,global_lsl_offset_unresolved,The repaired file uses IDUN raw time and conta...
92,ImBrAid,Z8HDMS,VX,error,source_discovery_failed,Discovery status: missing_raw
93,ImBrAid,ZQTXLD,VX,warning,xdf_chunk_reordering,401 adjacent XDF chunk transitions were not ch...
94,ImBrAid,ZQTXLD,VX,warning,nonmonotonic_xdf_timestamps,21439 uncorrected XDF sample intervals were no...
